In [3]:
import json
from pathlib import Path
from collections import Counter, defaultdict

JSONL_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_1_LLM_summary"
)
MONTH = "adzuna_month01_npz"  # change if needed
jsonl_dir = JSONL_ROOT / MONTH

# what we expect to be siblings
CATEGORY_KEYS = {
    "job_sector_category", "job_category", "category", "sector", "industry",
    "jobSectorCategory", "sector_category", "sectorCategory"
}
DESC_KEYS = {"job_description", "description", "full_text", "text"}

top_level_key_hits = Counter()
nested_key_hits = Counter()
sibling_stats = Counter()
examples = []

def flatten_keys(obj, prefix=""):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else k
            out.append(p)
            out.extend(flatten_keys(v, p))
    return out

def is_nonempty(x):
    if x is None:
        return False
    if isinstance(x, str):
        return x.strip() != ""
    return True

n_rows = 0
n_with_desc = 0
n_with_any_cat_top = 0
n_with_any_cat_anywhere = 0
n_with_desc_and_cat_as_siblings = 0

for fp in sorted(jsonl_dir.glob("*.jsonl")):
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except Exception:
                continue

            n_rows += 1

            # top-level keys
            if isinstance(obj, dict):
                for k in obj.keys():
                    top_level_key_hits[k] += 1

            # detect top-level description key present
            has_desc_top = any(k in obj and is_nonempty(obj.get(k)) for k in DESC_KEYS) if isinstance(obj, dict) else False
            if has_desc_top:
                n_with_desc += 1

            # detect top-level category key present
            has_cat_top = any(k in obj and is_nonempty(obj.get(k)) for k in CATEGORY_KEYS) if isinstance(obj, dict) else False
            if has_cat_top:
                n_with_any_cat_top += 1

            # detect category anywhere (flatten)
            flat = flatten_keys(obj)
            for k in flat:
                if any(tok in k.lower() for tok in ["category", "sector", "industry"]):
                    nested_key_hits[k] += 1

            has_cat_anywhere = any(
                any(tok in k.lower() for tok in ["category", "sector", "industry"])
                for k in flat
            )
            if has_cat_anywhere:
                n_with_any_cat_anywhere += 1

            # sibling check: description and category both top-level
            if has_desc_top and has_cat_top:
                n_with_desc_and_cat_as_siblings += 1

            # collect a few examples that have a description but no top-level category
            if has_desc_top and not has_cat_top and len(examples) < 5 and isinstance(obj, dict):
                keep = {k: obj.get(k) for k in obj.keys() if k in (DESC_KEYS | {"job_id","id","jobid","title","job_ad_title","domain","meta"})}
                examples.append(keep)

print(f"DIR: {jsonl_dir}")
print(f"ROWS parsed: {n_rows:,}")
print(f"Rows with TOP-LEVEL description: {n_with_desc:,} ({n_with_desc/max(n_rows,1):.1%})")
print(f"Rows with TOP-LEVEL category/sector: {n_with_any_cat_top:,} ({n_with_any_cat_top/max(n_rows,1):.1%})")
print(f"Rows with ANY category-like key anywhere: {n_with_any_cat_anywhere:,} ({n_with_any_cat_anywhere/max(n_rows,1):.1%})")
print(f"Rows with desc+category as TOP-LEVEL siblings: {n_with_desc_and_cat_as_siblings:,} ({n_with_desc_and_cat_as_siblings/max(n_rows,1):.1%})")

print("\nTop-level keys (most common):")
for k,c in top_level_key_hits.most_common(40):
    print(f"{c:>8}  {k}")

print("\nNested category-like keypaths (most common):")
for k,c in nested_key_hits.most_common(40):
    print(f"{c:>8}  {k}")

print("\nExample rows (desc present, no top-level category):")
for i,e in enumerate(examples, 1):
    print(f"\n--- example {i} ---")
    print(json.dumps(e, ensure_ascii=False)[:1500])


DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/adzuna_month01_npz
ROWS parsed: 2,658,928
Rows with TOP-LEVEL description: 2,658,928 (100.0%)
Rows with TOP-LEVEL category/sector: 0 (0.0%)
Rows with ANY category-like key anywhere: 2,658,928 (100.0%)
Rows with desc+category as TOP-LEVEL siblings: 0 (0.0%)

Top-level keys (most common):
 2658928  id
 2658928  category_name
 2658928  title
 2658928  job_description
 2658928  llm_output
 2658928  parsed
 2658928  error

Nested category-like keypaths (most common):
 2658928  category_name

Example rows (desc present, no top-level category):

--- example 1 ---
{"id": "2746717438", "title": "Site Bench Officer", "job_description": "A fantastic Opportunity has arisen within Bidvest Noonan for Site Bench Officers , who will work at our site in Battersea. You must be able to demonstrate and deliver high quality guarding and excellent customer service to both our clients and service users. We re

In [2]:
import json
from pathlib import Path

p = next(Path("/projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/adzuna_month01_npz").glob("*.jsonl"))
line = next(p.open("r", encoding="utf-8"))
obj = json.loads(line)

print("keys:", sorted(obj.keys()))
print("id:", obj.get("id"))
print("title:", obj.get("title"))
print("job_description exists:", "job_description" in obj)
print("category_name:", obj.get("category_name"))


keys: ['category_name', 'error', 'id', 'job_description', 'llm_output', 'parsed', 'title']
id: 2841472113
title: Customer Service Executive
job_description exists: True
category_name: Customer Services Jobs


In [ ]:
    job_ids = data["job_ids"]
    job_desc = data["job_desc"]
    job_tasks = data["job_tasks"]
    domains = data["domain"]
    candidates = data["titles"]
    job_ad_titles = data["job_ad_title"]
    job_sector_categories = data["job_sector_category"]
    job_full_ads = data["job_description"]